# Day 3: Guided Lab — Building a RAG Pipeline

## Learning Objectives

1. Create text embeddings using the Gemini Embeddings API
2. Implement cosine similarity and nearest-neighbor search
3. Split documents into semantically meaningful chunks
4. Build a complete RAG pipeline (retrieve → augment → generate)
5. Evaluate RAG outputs using the RAG triad framework

## Prerequisites

- [x] Completed Day 2 labs
- [x] Understanding of embeddings from input session
- [x] Google Colab with `GEMINI_API_KEY` configured

---
## Part 0: Setup and Infrastructure

In [ ]:
!pip install -q -U google-genai

In [ ]:
# ── Imports ──────────────────────────────────────────────
import os, time, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

from google import genai
from google.genai import types

# ── API Key ───────────────────────────────────────────────
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

MODEL_ID = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"

print(f"API key loaded: {'yes' if API_KEY else 'no'}")
print(f"Generation model: {MODEL_ID}")
print(f"Embedding model:  {EMBEDDING_MODEL}")

In [ ]:
# ── Logging Infrastructure ────────────────────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, temperature=0.7, max_tokens=1000, log=True, label=None):
    """Generate free-form text. Returns raw string."""
    t0 = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"temperature": temperature, "max_output_tokens": max_tokens},
    )
    latency = time.time() - t0
    text = response.text
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(), "label": label or "generate",
            "type": "free_form",
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": text[:300] + "..." if len(text) > 300 else text,
            "response_length": len(text),
            "latency_s": round(latency, 2),
        })
    return text

def generate_structured(prompt, schema_model, temperature=0.2, log=True, label=None):
    """Generate structured JSON output. Returns Pydantic model instance."""
    t0 = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_schema": schema_model,
        },
    )
    latency = time.time() - t0
    result = schema_model.model_validate_json(response.text)
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(), "label": label or "structured",
            "type": "structured",
            "schema": schema_model.__name__,
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": response.text[:300],
            "response_length": len(response.text),
            "latency_s": round(latency, 2),
        })
    return result

def show_log():
    """Display prompt log as DataFrame."""
    if not PROMPT_LOG:
        print("No API calls logged yet.")
        return None
    return pd.DataFrame(PROMPT_LOG)

print("Logging infrastructure ready.")

In [ ]:
# ── RAG Infrastructure ────────────────────────────────────

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """
    Embed one or more texts using Gemini Embeddings API.
    
    Args:
        texts: A string or list of strings to embed.
        task_type: 'RETRIEVAL_DOCUMENT', 'RETRIEVAL_QUERY', or 'SEMANTIC_SIMILARITY'.
    
    Returns:
        List of numpy arrays (one embedding per input text).
    """
    if isinstance(texts, str):
        texts = [texts]
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return [np.array(e.values) for e in response.embeddings]


def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search(query, doc_embeddings, documents, top_k=3):
    """
    Find top-k most similar documents to a query.
    
    Returns: list of (index, score, document_text) tuples, sorted by score descending.
    """
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    scores = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_emb, doc_emb)
        scores.append((i, sim, documents[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


def rag_query(question, documents, doc_embeddings, top_k=3, system_prompt=None):
    """
    Complete RAG pipeline: retrieve → augment → generate.
    
    Returns: (answer_text, retrieved_results)
    """
    # Retrieve
    results = search(question, doc_embeddings, documents, top_k)
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{doc}" for i, (_, _, doc) in enumerate(results)
    )
    
    # Augment
    if system_prompt is None:
        system_prompt = (
            "You are a helpful assistant that answers questions based on "
            "the provided context. If the answer is not in the context, "
            'say "I don\'t have enough information to answer this." '
            "Do not make up information."
        )
    
    prompt = f"""{system_prompt}

<context>
{context}
</context>

Question: {question}"""
    
    # Generate
    answer = generate(prompt, temperature=0.3, label=f"rag_{question[:30]}")
    return answer, results

print("RAG infrastructure ready: embed_texts, cosine_similarity, search, rag_query")

---
## Part 1: Text Embeddings

Embeddings convert text into numerical vectors where **similar meanings are close together**.
We'll use Google's `gemini-embedding-001` model which produces 3072-dimensional vectors.

### Exercise 1.1: Embed Documents

Let's embed a set of company documents and examine what the embedding vectors look like.

In [ ]:
# Our sample knowledge base (6 company documents)
documents = [
    "The company was founded in 2015 by Maria Chen and David Park. "
    "It started as a two-person startup in a garage in Munich.",

    "Our remote work policy allows employees to work from home up to "
    "3 days per week. All team meetings on Tuesdays and Thursdays are mandatory in-person.",

    "Q3 2025 revenue was EUR 4.2 million, a 15% increase over Q3 2024. "
    "Growth was primarily driven by the enterprise segment.",

    "The company offers 30 days of annual leave plus public holidays. "
    "Unused leave can be carried over for up to 6 months.",

    "Our main product, DataSync Pro, integrates with SAP, Salesforce, "
    "and Microsoft 365. The API supports REST and GraphQL.",

    "The engineering team follows a two-week sprint cycle with planning "
    "on Mondays and retrospectives on Fridays.",
]

# Embed all documents
doc_embeddings = embed_texts(documents, task_type="RETRIEVAL_DOCUMENT")

print(f"Number of documents: {len(doc_embeddings)}")
print(f"Embedding dimension: {len(doc_embeddings[0])}")
print(f"\nFirst 5 values of document 0: {doc_embeddings[0][:5]}")
print(f"Vector norm (length):          {np.linalg.norm(doc_embeddings[0]):.4f}")

### Exercise 1.2: Compare Task Types

The Gemini Embeddings API supports different **task types** that optimise the embedding for its intended use.

In [ ]:
query_text = "Can I work from home?"

# Embed the same text with different task types
emb_doc   = embed_texts(query_text, task_type="RETRIEVAL_DOCUMENT")[0]
emb_query = embed_texts(query_text, task_type="RETRIEVAL_QUERY")[0]
emb_sim   = embed_texts(query_text, task_type="SEMANTIC_SIMILARITY")[0]

print(f"Same text, different task types:")
print(f"  DOC  vs QUERY:      cosine = {cosine_similarity(emb_doc, emb_query):.4f}")
print(f"  DOC  vs SIMILARITY: cosine = {cosine_similarity(emb_doc, emb_sim):.4f}")
print(f"  QUERY vs SIMILARITY: cosine = {cosine_similarity(emb_query, emb_sim):.4f}")
print()
print("Notice: The same text produces DIFFERENT vectors depending on task type!")
print("Use RETRIEVAL_DOCUMENT for documents, RETRIEVAL_QUERY for queries.")

> **💡 Discussion:** Each embedding is a 3072-dimensional vector. You can think of each dimension as capturing one aspect of meaning. Similar texts will have similar patterns across all 3072 dimensions, which is why cosine similarity works — it measures how much two vectors "point in the same direction" in this high-dimensional space.

---
## Part 2: Similarity Search

Now that we have embeddings, we can find documents by **meaning** instead of keywords.

### Exercise 2.1: Pairwise Similarity

Let's see how similar our 6 documents are to each other.

In [ ]:
# Compute pairwise cosine similarity
n = len(doc_embeddings)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(doc_embeddings[i], doc_embeddings[j])

# Display as DataFrame
labels = [f"Doc {i}" for i in range(n)]
sim_df = pd.DataFrame(sim_matrix, index=labels, columns=labels).round(3)
print("Pairwise cosine similarity between documents:")
print(sim_df.to_string())
print()

# Find most similar pair (excluding self-comparisons)
np.fill_diagonal(sim_matrix, 0)
max_idx = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f"Most similar pair: Doc {max_idx[0]} and Doc {max_idx[1]} "
      f"(cosine = {sim_matrix[max_idx]:.3f})")
print(f"  Doc {max_idx[0]}: {documents[max_idx[0]][:60]}...")
print(f"  Doc {max_idx[1]}: {documents[max_idx[1]][:60]}...")

### Exercise 2.2: Semantic Search

Search our documents by meaning using the `search()` function.

In [ ]:
queries = [
    "Can I work from home?",
    "How much money did we make?",
    "Who started the company?",
    "What tools does the product connect to?",
]

for q in queries:
    results = search(q, doc_embeddings, documents, top_k=2)
    print(f"\nQuery: \"{q}\"")
    for idx, score, doc in results:
        print(f"  [{score:.3f}] Doc {idx}: {doc[:70]}...")

---
## Part 3: Document Chunking

Real documents are long. We need to **split them into smaller chunks** before embedding.
Let's explore different chunking strategies.

### Exercise 3.1: Fixed-Size Chunking

In [ ]:
# A longer document to chunk
long_document = """Remote Work Policy (Version 4.0, Updated January 2025)

Section 1: General Guidelines
Our company supports flexible work arrangements to help employees balance 
productivity and well-being. Employees may work remotely up to 3 days per 
week, subject to manager approval. New employees in their first 90 days 
must work on-site full-time to complete onboarding.

Section 2: Mandatory In-Office Days
All team-wide meetings are held on Tuesdays and Thursdays. Attendance is 
mandatory and in-person. Department heads may designate additional in-office 
days for project-critical phases. Failure to attend mandatory meetings 
without prior approval will be noted in performance reviews.

Section 3: Equipment and Expenses
The company provides a one-time home office stipend of EUR 500 for ergonomic 
equipment (desk, chair, monitor). Internet costs are reimbursed up to 
EUR 30 per month upon submission of receipts. All company equipment 
must be returned upon termination.

Section 4: Security Requirements
Remote workers must use the company VPN at all times when accessing 
internal systems. Sensitive documents must not be printed at home. 
Screen locks must engage after 5 minutes of inactivity. Any security 
incidents must be reported to IT within 1 hour."""

# Fixed-size chunking (by word count)
def chunk_fixed(text, chunk_size=50, overlap=10):
    """Split text into chunks of approximately chunk_size words with overlap."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Overlap
    return chunks

fixed_chunks = chunk_fixed(long_document, chunk_size=50, overlap=10)
print(f"Fixed-size chunking: {len(fixed_chunks)} chunks\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i} ({len(chunk.split())} words): {chunk[:80]}...")

### Exercise 3.2: Sentence-Based Chunking

In [ ]:
# Sentence-based chunking (respects sentence boundaries)
def chunk_sentences(text, max_words=60, overlap_words=15):
    """Split text at sentence boundaries, grouping sentences up to max_words."""
    # Simple sentence split (handles common abbreviations poorly, but fine for demo)
    sentences = [s.strip() for s in text.replace('\n', ' ').split('. ') if s.strip()]
    
    chunks = []
    current = []
    current_len = 0
    
    for sent in sentences:
        sent_words = len(sent.split())
        if current_len + sent_words > max_words and current:
            chunks.append(". ".join(current) + ".")
            # Keep last sentence(s) as overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                if overlap_len + len(s.split()) <= overlap_words:
                    overlap_sents.insert(0, s)
                    overlap_len += len(s.split())
                else:
                    break
            current = overlap_sents
            current_len = overlap_len
        current.append(sent)
        current_len += sent_words
    
    if current:
        chunks.append(". ".join(current) + ".")
    return chunks

sent_chunks = chunk_sentences(long_document, max_words=60, overlap_words=15)
print(f"Sentence-based chunking: {len(sent_chunks)} chunks\n")
for i, chunk in enumerate(sent_chunks):
    print(f"Chunk {i} ({len(chunk.split())} words):")
    print(f"  {chunk[:120]}...\n")

### Exercise 3.3: Adding Chunk Metadata

In production, you need to know **where each chunk came from** — which document, which section, when it was last updated. Let's attach metadata to our chunks.

In [ ]:
# Attach metadata to each chunk (production pattern)
from dataclasses import dataclass

@dataclass
class Chunk:
    """A chunk with metadata for traceability."""
    chunk_id: str
    doc_title: str
    text: str

# Create metadata-aware chunks from our policy document
policy_title = "Remote Work Policy v4.0"
metadata_chunks = []
for i, text in enumerate(sent_chunks):
    metadata_chunks.append(Chunk(
        chunk_id=f"REMOTE_WORK::C{i+1}",
        doc_title=policy_title,
        text=text,
    ))

print(f"Created {len(metadata_chunks)} chunks with metadata:\n")
for c in metadata_chunks:
    print(f"  {c.chunk_id} ({c.doc_title})")
    print(f"    {c.text[:80]}...\n")

print("Now each chunk carries its identity — essential for citations and debugging.")

> **💡 Discussion: Chunk Size Trade-offs**
>
> | | Small Chunks (~50 words) | Large Chunks (~150 words) |
> |---|---|---|
> | **Precision** | High — focused content | Lower — includes noise |
> | **Context** | Risk missing surrounding info | Better coverage |
> | **Number** | Many chunks to search | Fewer, more efficient |
>
> **Rule of thumb:** Start with 200–500 tokens (~50–125 words) with 10–20% overlap.

---
## Part 4: The Complete RAG Pipeline

Now let's put it all together: **retrieve** relevant chunks, **augment** the prompt, and **generate** a grounded answer.

### Exercise 4.1: RAG with Grounding Rules

In [ ]:
# Use the sentence-based chunks from our remote work policy
policy_chunks = sent_chunks
policy_embeddings = embed_texts(policy_chunks, task_type="RETRIEVAL_DOCUMENT")

# Define a strong grounding prompt
SYSTEM_PROMPT = """You are a company policy assistant.
Answer questions based ONLY on the provided policy excerpts.
Rules:
- If the answer is not in the excerpts, say: "I don't have that information in the provided policies."
- Cite which chunk(s) support your answer using [Chunk N] notation.
- Keep answers concise (under 80 words).
- Do not invent policy details."""

# Ask a question
question = "How many days can I work from home?"
answer, retrieved = rag_query(
    question, policy_chunks, policy_embeddings,
    top_k=3, system_prompt=SYSTEM_PROMPT
)

print(f"Question: {question}\n")
print("Retrieved chunks:")
for idx, score, doc in retrieved:
    print(f"  [{score:.3f}] Chunk {idx}: {doc[:70]}...")
print(f"\nAnswer:\n{answer}")

### Exercise 4.2: With vs. Without Context (Hallucination Demo)

In [ ]:
question = "What is the home office stipend amount?"

# WITH RAG (grounded)
rag_answer, _ = rag_query(
    question, policy_chunks, policy_embeddings,
    top_k=3, system_prompt=SYSTEM_PROMPT
)

# WITHOUT RAG (ungrounded — model must rely on "memory")
no_rag_answer = generate(
    f"Answer this company policy question: {question}",
    temperature=0.3, label="no_rag_comparison"
)

print(f"Question: {question}\n")
print(f"WITH RAG (grounded):\n{rag_answer}\n")
print(f"WITHOUT RAG (ungrounded):\n{no_rag_answer}\n")
print("Notice: Without context, the model either guesses or admits it doesn't know.")
print("With RAG, the answer is grounded in actual policy text.")

### Exercise 4.3: Out-of-Scope Question (Refusal Test)

In [ ]:
# This question has NO answer in our policy documents
out_of_scope = "What programming language is the backend written in?"

answer, retrieved = rag_query(
    out_of_scope, policy_chunks, policy_embeddings,
    top_k=3, system_prompt=SYSTEM_PROMPT
)

print(f"Question: {out_of_scope}\n")
print(f"Answer:\n{answer}\n")
print("The system should refuse to answer (not hallucinate a programming language).")

---
## Part 5: RAG Evaluation

Measure RAG quality systematically using a golden test set, keyword matching, retrieval metrics, and LLM-as-judge scoring.

### Exercise 5.1: Define a Golden Test Set

In [ ]:
# Golden set: questions with expected answers, source chunks, and expected chunk indices
GOLDEN_SET = [
    {
        "id": "Q1",
        "question": "How many days per week can I work remotely?",
        "expected_keywords": ["3 days", "three days"],
        "expected_chunks": [0, 1],  # Chunk indices that should be retrieved
        "difficulty": "easy",
    },
    {
        "id": "Q2",
        "question": "Which days must I be in the office?",
        "expected_keywords": ["Tuesday", "Thursday"],
        "expected_chunks": [1],
        "difficulty": "easy",
    },
    {
        "id": "Q3",
        "question": "How much is the home office equipment budget?",
        "expected_keywords": ["500", "EUR 500"],
        "expected_chunks": [2, 3],
        "difficulty": "easy",
    },
    {
        "id": "Q4",
        "question": "Can new employees work from home immediately?",
        "expected_keywords": ["90 days", "onboarding", "first 90"],
        "expected_chunks": [0, 1],
        "difficulty": "medium",
    },
    {
        "id": "Q5",
        "question": "What happens if I miss a mandatory meeting?",
        "expected_keywords": ["performance review"],
        "expected_chunks": [1, 2],
        "difficulty": "medium",
    },
    {
        "id": "Q6",
        "question": "How quickly must security incidents be reported?",
        "expected_keywords": ["1 hour", "one hour"],
        "expected_chunks": [3, 4],
        "difficulty": "medium",
    },
    {
        "id": "Q7",
        "question": "What is the company's revenue forecast for 2026?",
        "expected_keywords": ["don't have", "not in", "no information"],
        "expected_chunks": [],  # No relevant chunk exists
        "difficulty": "refusal",
    },
    {
        "id": "Q8",
        "question": "What's the best restaurant near the office?",
        "expected_keywords": ["don't have", "not in", "no information"],
        "expected_chunks": [],
        "difficulty": "refusal",
    },
]

print(f"Golden set: {len(GOLDEN_SET)} questions")
for q in GOLDEN_SET:
    chunks_str = ', '.join(str(c) for c in q['expected_chunks']) if q['expected_chunks'] else 'none'
    print(f"  [{q['difficulty']:7s}] {q['id']}: {q['question']}  (chunks: {chunks_str})")

### Exercise 5.2: Keyword-Based Evaluation

Run the RAG pipeline on all golden set questions and check whether the answers contain the expected keywords.

In [ ]:
# Run RAG on all golden set questions
rag_results = []

for qa in GOLDEN_SET:
    answer, retrieved = rag_query(
        qa["question"], policy_chunks, policy_embeddings,
        top_k=3, system_prompt=SYSTEM_PROMPT
    )
    
    # Check if answer contains expected keywords
    answer_lower = answer.lower()
    keyword_hit = any(
        kw.lower() in answer_lower for kw in qa["expected_keywords"]
    )
    
    rag_results.append({
        "id": qa["id"],
        "question": qa["question"],
        "difficulty": qa["difficulty"],
        "answer": answer,
        "keyword_match": keyword_hit,
        "top_chunk_score": retrieved[0][1],
    })

# Summary
results_df = pd.DataFrame(rag_results)
accuracy = results_df["keyword_match"].mean()
print(f"\nOverall keyword accuracy: {accuracy:.0%} ({results_df['keyword_match'].sum()}/{len(results_df)})")
print()
print(results_df[["id", "difficulty", "keyword_match", "top_chunk_score"]].to_string(index=False))

### Exercise 5.3: Precision@k and Recall@k

Beyond keyword matching, we can measure **retrieval quality** directly using standard IR metrics:

- **Precision@k** = What fraction of retrieved chunks were relevant?
- **Recall@k** = What fraction of relevant chunks were retrieved?

In [ ]:
# Precision@k and Recall@k
def precision_recall_at_k(retrieved_indices, expected_chunks, k):
    """Compute Precision@k and Recall@k for a single query."""
    retrieved_set = set(retrieved_indices[:k])
    expected_set = set(expected_chunks)
    if not expected_set:  # Refusal questions have no expected chunks
        return None, None
    hits = retrieved_set & expected_set
    precision = len(hits) / k if k > 0 else 0
    recall = len(hits) / len(expected_set) if expected_set else 0
    return precision, recall

# Compute for all non-refusal golden set questions
retrieval_metrics = []
for qa in GOLDEN_SET:
    if not qa["expected_chunks"]:  # Skip refusal questions
        continue
    # Get retrieved chunk indices
    results = search(qa["question"], policy_embeddings, policy_chunks, top_k=3)
    retrieved_indices = [idx for idx, _, _ in results]
    
    p, r = precision_recall_at_k(retrieved_indices, qa["expected_chunks"], k=3)
    retrieval_metrics.append({
        "id": qa["id"],
        "question": qa["question"][:40],
        "precision@3": p,
        "recall@3": r,
        "retrieved": retrieved_indices,
        "expected": qa["expected_chunks"],
    })

metrics_df = pd.DataFrame(retrieval_metrics)
print("Retrieval Metrics (Precision@k / Recall@k):")
print(metrics_df[["id", "precision@3", "recall@3"]].to_string(index=False))
print(f"\nAvg Precision@3: {metrics_df['precision@3'].mean():.2f}")
print(f"Avg Recall@3:    {metrics_df['recall@3'].mean():.2f}")

### Exercise 5.4: LLM-as-Judge Scoring

In [ ]:
# Use the LLM to score answer quality
class RAGScore(BaseModel):
    """RAG quality score for a single answer."""
    groundedness: int = Field(description="1-5: Is the answer supported by the context?")
    relevance: int = Field(description="1-5: Does the answer address the question?")
    explanation: str = Field(description="Brief explanation of the scores")

# Score a few answers
scores = []
for r in rag_results[:5]:  # Score first 5 to save API calls
    eval_prompt = f"""Rate this RAG system answer.

Question: {r['question']}
Answer: {r['answer']}

Score groundedness (1-5): Is the answer based on facts, not made up?
Score relevance (1-5): Does the answer address the question?"""
    
    score = generate_structured(eval_prompt, RAGScore, label=f"eval_{r['id']}")
    scores.append({
        "id": r["id"],
        "groundedness": score.groundedness,
        "relevance": score.relevance,
        "explanation": score.explanation,
    })

scores_df = pd.DataFrame(scores)
print("LLM-as-Judge Scores:")
print(scores_df.to_string(index=False))
print(f"\nAvg Groundedness: {scores_df['groundedness'].mean():.1f}/5")
print(f"Avg Relevance:    {scores_df['relevance'].mean():.1f}/5")

---
## Summary and Key Takeaways

| Technique | What You Learned | Key Function |
|-----------|-----------------|---------------|
| **Text Embeddings** | Convert text to 3072-dim vectors | `embed_texts()` |
| **Task Types** | RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY | `task_type` parameter |
| **Cosine Similarity** | Measure semantic closeness (0–1) | `cosine_similarity()` |
| **Document Chunking** | Split long docs for precise retrieval | `chunk_fixed()`, `chunk_sentences()` |
| **Chunk Metadata** | Attach doc_id, title for traceability | `@dataclass Chunk` |
| **RAG Pipeline** | Retrieve → Augment → Generate | `rag_query()` |
| **Grounding Prompts** | Force model to use only context | System prompt rules |
| **Golden Test Set** | Systematic evaluation of RAG quality | Expected keywords + LLM-as-judge |
| **Precision@k / Recall@k** | Measure retrieval quality directly | Standard IR metrics |

### Checklist

- [x] Embedded documents and examined vector properties
- [x] Compared RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY task types
- [x] Computed pairwise similarity between documents
- [x] Searched by meaning (semantic search)
- [x] Chunked a longer document two ways
- [x] Attached metadata to chunks for traceability
- [x] Built a complete RAG pipeline with grounding prompt
- [x] Tested with vs. without context (hallucination demo)
- [x] Tested refusal on out-of-scope questions
- [x] Evaluated with golden test set and keyword matching
- [x] Computed Precision@k and Recall@k retrieval metrics
- [x] Scored answers with LLM-as-judge

In [ ]:
# ── Export Experiment Log ──────────────────────────────────
if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day3_guided_lab_log.csv", index=False)
    print(f"Exported {len(PROMPT_LOG)} API calls to day3_guided_lab_log.csv")
    print(log_df[["label", "type", "latency_s"]].to_string(index=False))

---
## Next Steps

In the **Independent Lab**, you will:
- Choose a business domain (HR, Product Docs, Support, or Research)
- Build your own RAG knowledge base
- Create a golden test set of 10+ questions
- Iterate your prompt (v1 → v2) with measured improvement
- Write an error analysis

→ Proceed to the [Independent Lab](03-03_lab_2.qmd)